In [1]:
pip install fastapi uvicorn python-multipart pdfplumber python-docx pytesseract pillow pdf2image

   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ---------------------------------------- 4.0/4.0 MB 40.0 MB/s  0:00:00

   ------------- -------------------------- 1/3 [lxml]
   ------------- -------------------------- 1/3 [lxml]
   -------------------------- ------------- 2/3 [python-docx]
   -------------------------- ------------- 2/3 [python-docx]
   -------------------------- ------------- 2/3 [python-docx]
   ---------------------------------------- 3/3 [python-docx]

Note: you may need to restart the kernel to use updated packages.


## File reader(Core Extraction)

#### 👉 app/utils/file_reader.py

In [2]:
import pdfplumber
import pytesseract
from pdf2image import convert_from_path
from docx import Document
from PIL import Image
import os


# -------------------------
# PDF TEXT EXTRACTION
# -------------------------
def extract_from_pdf(file_path):
    text = ""

    try:
        with pdfplumber.open(file_path) as pdf:
            for page in pdf.pages:
                text += page.extract_text() or ""
    except:
        pass

    # If PDF is scanned → fallback OCR
    if len(text.strip()) == 0:
        images = convert_from_path(file_path)
        for img in images:
            text += pytesseract.image_to_string(img)

    return text


# -------------------------
# WORD FILE EXTRACTION
# -------------------------
def extract_from_docx(file_path):
    doc = Document(file_path)
    return "\n".join([p.text for p in doc.paragraphs])


# -------------------------
# IMAGE OCR
# -------------------------
def extract_from_image(file_path):
    img = Image.open(file_path)
    text = pytesseract.image_to_string(img)
    return text


# -------------------------
# MAIN FUNCTION
# -------------------------
def extract_text(file_path):
    ext = os.path.splitext(file_path)[1].lower()

    if ext == ".pdf":
        return extract_from_pdf(file_path)

    elif ext == ".docx":
        return extract_from_docx(file_path)

    elif ext in [".png", ".jpg", ".jpeg"]:
        return extract_from_image(file_path)

    else:
        raise ValueError("Unsupported file type")

## Text Cleaning

### 👉 app/utils/text_cleaning.py

In [3]:
import re

def clean_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^a-zA-Z0-9?.!, ]", "", text)
    return text.strip()

## JSON Formatter

### 👉 app/utils/json_formatter.py

In [4]:
import uuid
from datetime import datetime

def format_question(text, topic=None, source="upload"):
    return {
        "id": str(uuid.uuid4()),
        "text": text,
        "topic": topic,
        "source": source,
        "created_at": datetime.utcnow().isoformat()
    }

## CSV Handler

### 👉 app/utils/csv_handler.py

In [5]:
import csv
import os

CSV_PATH = "data/processed/questions.csv"


def append_to_csv(question_json):
    file_exists = os.path.isfile(CSV_PATH)

    with open(CSV_PATH, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=question_json.keys())

        if not file_exists:
            writer.writeheader()

        writer.writerow(question_json)

## Ingestion Service(Core Pipeline)

### 👉 app/services/ingestion_service.py

In [6]:
import os

from app.utils.file_reader import extract_text
from app.utils.text_cleaning import clean_text
from app.utils.json_formatter import format_question
from app.utils.csv_handler import append_to_csv

DATA_PATH = "data/processed/questions.json"


def load_json():
    if not os.path.exists(DATA_PATH):
        return []
    import json
    with open(DATA_PATH, "r") as f:
        return json.load(f)


def save_json(data):
    import json
    with open(DATA_PATH, "w") as f:
        json.dump(data, f, indent=4)


def process_file(file_path):
    raw_text = extract_text(file_path)

    # split into possible questions
    lines = raw_text.split("\n")

    results = []

    data = load_json()

    for line in lines:
        cleaned = clean_text(line)

        if len(cleaned) < 5:
            continue

        question_json = format_question(cleaned)

        # Save JSON
        data.append(question_json)

        # Append CSV
        append_to_csv(question_json)

        results.append(question_json)

    save_json(data)

    return results

c:\Users\Ismat\anaconda3\envs\rec_env\lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


FileNotFoundError: [Errno 2] No such file or directory: 'data/questions.json'

## FastAPI Upload Endpoint

### 👉 app/api/routes_upload.py

In [7]:
from fastapi import APIRouter, UploadFile, File
import os

from app.services.ingestion_service import process_file

router = APIRouter()

UPLOAD_DIR = "data/uploads"


@router.post("/upload-file")
def upload_file(file: UploadFile = File(...)):

    os.makedirs(UPLOAD_DIR, exist_ok=True)

    file_path = os.path.join(UPLOAD_DIR, file.filename)

    with open(file_path, "wb") as f:
        f.write(file.file.read())

    results = process_file(file_path)

    return {
        "message": "File processed successfully",
        "questions_extracted": len(results)
    }

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


FileNotFoundError: [Errno 2] No such file or directory: 'data/questions.json'